In [1]:
# --- Pré-requisito: namespaces já devem existir (rodado uma vez) ---
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

DataFrame[]

In [15]:
# --- Camada bronze: ingestão bruta, sem transformação ---
df_bronze = spark.read.option("header", True).csv("/home/iceberg/data/raw/vendas_detalhadas.csv")

df_bronze.writeTo("nessie.bronze.vendas").createOrReplace()

print("bronze:", spark.table("nessie.bronze.vendas").count(), "linhas")
spark.table("nessie.bronze.vendas").show()

bronze: 1000 linhas
+--------+--------------------+--------------------+------+------------------+-----------+----------+--------------+----------+
|id_venda|        cliente_nome|       cliente_email|estado|           produto|  categoria|quantidade|preco_unitario|data_venda|
+--------+--------------------+--------------------+------+------------------+-----------+----------+--------------+----------+
|       1|        Brenda Alves|samuel32@example.net|    AM|     Mouse sem Fio| Acessórios|         3|         120.0|2026-05-27|
|       2|       Aurora Pastor|bandrade@example.org|    DF|     Monitor 27pol|Eletrônicos|         3|        1300.0|2025-12-16|
|       3|       Amanda Novais| zoeleao@example.org|    PB|Cadeira Ergonômica|     Móveis|         3|         950.0|2026-04-25|
|       4|     Emanuel Sampaio|rfernandes@exampl...|    DF|    Notebook Gamer|Eletrônicos|         3|        4500.0|2026-09-07|
|       5|    Sr. Caleb Garcia|fogacacatarina@ex...|    GO|     Monitor 27pol|Eletrô

In [20]:
from pyspark.sql.functions import col, to_date                                                                  
                                                                                                                    
df_bronze = spark.table("nessie.bronze.vendas")                                                      
                                                                                                                    
df_silver = (                                                                                                   
    df_bronze                                                                                                   
    .withColumn("id_venda", col("id_venda").cast("long"))                                                       
    .withColumn("quantidade", col("quantidade").cast("integer"))                                                
    .withColumn("preco_unitario", col("preco_unitario").cast("decimal(10,2)"))                                  
    .withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))                                         
    .withColumn("valor_total", col("quantidade") * col("preco_unitario"))                                       
    .dropDuplicates(["id_venda"]) # Deduplicação por chave de negócio                                           
)                                                                                                               
                                                                                                                    
# Gravar particionando por estado na Silver                                                                     
df_silver.writeTo("nessie.silver.vendas").partitionedBy(col("estado")).createOrReplace()                                                                                          
                                                                                                                    
print("Silver criada com sucesso!")                                                                             
spark.table("nessie.silver.vendas").printSchema()  

Silver criada com sucesso!
root
 |-- id_venda: long (nullable = true)
 |-- cliente_nome: string (nullable = true)
 |-- cliente_email: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: decimal(10,2) (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- valor_total: decimal(21,2) (nullable = true)



In [21]:
from pyspark.sql.functions import sum as _sum, count as _count, round as _round                                 
                                                                                                                    
df_silver = spark.table("nessie.silver.vendas")                                                      
                                                                                                                    
# Exemplo de Gold: Faturamento e quantidade de pedidos por Categoria e Estado                                   
df_gold_categoria = (                                                                                           
    df_silver                                                                                                   
    .groupBy("estado", "categoria")                                                                             
    .agg(                                                                                                       
        _count("id_venda").alias("total_pedidos"),                                                              
        _round(_sum("valor_total"), 2).alias("receita_total")                                                   
    )                                                                                                           
    .orderBy("estado", "receita_total", ascending=False)                                                        
)                                                                                                               
                                                                                                                    
df_gold_categoria.writeTo("nessie.gold.faturamento_por_categoria").createOrReplace()                            
                                                                                                                    
print("Gold finalizada:")                                                                                       
spark.table("nessie.gold.faturamento_por_categoria").show(10)   

Gold finalizada:
+------+-----------+-------------+-------------+
|estado|  categoria|total_pedidos|receita_total|
+------+-----------+-------------+-------------+
|    TO|Eletrônicos|           17|    133500.00|
|    TO|     Móveis|            5|     10450.00|
|    TO| Acessórios|           10|      3680.00|
|    SP|Eletrônicos|            8|     61900.00|
|    SP|     Móveis|           16|     38950.00|
|    SP| Acessórios|           11|      7080.00|
|    SE|Eletrônicos|            9|     78600.00|
|    SE|     Móveis|           10|     29450.00|
|    SE| Acessórios|            9|      6140.00|
|    SC|Eletrônicos|           14|    147500.00|
+------+-----------+-------------+-------------+
only showing top 10 rows



In [8]:
# --- Camada silver: limpeza, tipagem e deduplicação ---
df_silver = (
    df_raw
    .withColumn("valor", df_raw["valor"].cast("decimal(10,2)"))
    .dropDuplicates()
)

# checagem antes de gravar: cast falho vira NULL silenciosamente
nulos = df_silver.filter("valor IS NULL").count()
if nulos > 0:
    print(f"aviso: {nulos} linha(s) com 'valor' não numérico viraram NULL no cast")

df_silver.writeTo("nessie.silver.vendas").createOrReplace()

print("silver:", spark.table("nessie.silver.vendas").count(), "linhas")
spark.table("nessie.silver.vendas").show()

silver: 3 linhas
+--------+-------+------+
|  regiao|produto| valor|
+--------+-------+------+
|Nordeste|      A|150.50|
|     Sul|      B| 89.90|
| Sudeste|      C|320.00|
+--------+-------+------+



In [4]:
# --- Camada gold: agregação para consumo analítico ---
from pyspark.sql.functions import sum as _sum

df_gold = (
    df_silver
    .groupBy("regiao")
    .agg(_sum("valor").alias("total_vendas"))
    .orderBy("total_vendas", ascending=False)
)

df_gold.writeTo("nessie.gold.vendas_por_regiao").createOrReplace()

print("gold:", spark.table("nessie.gold.vendas_por_regiao").count(), "linhas")
spark.table("nessie.gold.vendas_por_regiao").show()

gold: 3 linhas
+--------+------------+
|  regiao|total_vendas|
+--------+------------+
| Sudeste|      320.00|
|Nordeste|      150.50|
|     Sul|       89.90|
+--------+------------+

